# 🏠 IWASAKI 3DGS STUDIO

**写真から「歩ける」3D空間を作る**

---

## 3D Gaussian Splatting (3DGS) とは

- 写真10-30枚から3Dシーンを再構築
- リアルタイムで視点を変えられる
- NeRFより高速、VRにも対応

**内装業での使い道：**
- 施工現場を3D化してお客様に見せる
- Before/Afterを「歩いて」比較できる
- VR内覧会

---

**A100 40GB専用。**

In [ ]:
#@title 🚀 **Step 0: GPU確認**

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                       capture_output=True, text=True)
gpu_info = result.stdout.strip()
print(f"\n🎮 GPU: {gpu_info}\n")

import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"💾 VRAM: {vram:.1f} GB")
    if 'A100' in gpu_info:
        print("\n✅ A100確認！3DGS学習も推論も余裕")
    else:
        print("\n⚠️ A100推奨。他のGPUでも動くが遅い")

In [ ]:
#@title 📦 **Step 1: 環境構築**
#@markdown 約5分かかる

# 基本パッケージ
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy scipy matplotlib pillow tqdm plyfile

# COLMAP（Structure from Motion）
!apt-get install -y colmap > /dev/null 2>&1

# gaussian-splatting 公式リポジトリ
!git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive
%cd gaussian-splatting

# 依存関係
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

print("\n✅ 環境構築完了！")

In [ ]:
#@title 📁 **Step 2: サンプルデータをダウンロード（またはアップロード）**
#@markdown まずはサンプルで試す。自分のデータは後で

import os

# サンプルデータ（Mip-NeRF 360のBicycle）
!mkdir -p /content/data
%cd /content/data

# 小さいサンプル（counter scene）
!wget -q https://storage.googleapis.com/gresearch/refraw360/360_v2_samples.zip
!unzip -q 360_v2_samples.zip -d samples

print("\n✅ サンプルデータ準備完了！")
print("\n📁 /content/data/samples/ に以下のシーンがあります：")
!ls /content/data/samples/

In [ ]:
#@title 📸 **Step 3: 自分の写真をアップロード（オプション）**
#@markdown 左のファイルパネルから /content/my_scene/input にアップロード

import os

os.makedirs('/content/my_scene/input', exist_ok=True)

print("📁 以下のフォルダに写真をアップロードしてください：")
print("   /content/my_scene/input/")
print("\n📸 撮影のコツ：")
print("   - 10-30枚（多いほど良い）")
print("   - いろんな角度から")
print("   - オーバーラップ確保（隣と30%以上重なる）")
print("   - ブレないように")
print("   - 明るさを一定に")
print("\nアップロードしたら次のセルでCOLMAP実行")

In [ ]:
#@title 🔧 **Step 4: COLMAP（カメラ位置推定）**
#@markdown 写真からカメラ位置を計算。約10-20分

use_sample = True #@param {type:"boolean"}
sample_scene = "counter" #@param ["bicycle", "bonsai", "counter", "flowers", "garden", "kitchen", "room", "stump", "treehill"]

if use_sample:
    scene_path = f"/content/data/samples/{sample_scene}"
    print(f"\n📁 サンプルシーン使用: {sample_scene}")
    print("   （サンプルはCOLMAP済みなのでスキップ）")
else:
    scene_path = "/content/my_scene"
    input_path = f"{scene_path}/input"
    
    # 写真数確認
    images = [f for f in os.listdir(input_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"\n📸 {len(images)}枚の写真を検出")
    
    if len(images) < 5:
        print("⚠️ 写真が少なすぎます。最低10枚推奨")
    else:
        print("\n🔄 COLMAP実行中...（10-20分）")
        
        os.makedirs(f"{scene_path}/distorted/sparse", exist_ok=True)
        
        # Feature extraction
        !colmap feature_extractor \
            --database_path {scene_path}/distorted/database.db \
            --image_path {input_path} \
            --ImageReader.single_camera 1 \
            --ImageReader.camera_model OPENCV \
            --SiftExtraction.use_gpu 1
        
        # Matching
        !colmap exhaustive_matcher \
            --database_path {scene_path}/distorted/database.db \
            --SiftMatching.use_gpu 1
        
        # Mapping
        !colmap mapper \
            --database_path {scene_path}/distorted/database.db \
            --image_path {input_path} \
            --output_path {scene_path}/distorted/sparse
        
        # Undistort
        !colmap image_undistorter \
            --image_path {input_path} \
            --input_path {scene_path}/distorted/sparse/0 \
            --output_path {scene_path} \
            --output_type COLMAP
        
        print("\n✅ COLMAP完了！")

In [ ]:
#@title 🏋️ **Step 5: 3DGS学習**
#@markdown A100で約10-30分

%cd /content/gaussian-splatting

use_sample = True #@param {type:"boolean"}
sample_scene = "counter" #@param ["bicycle", "bonsai", "counter", "flowers", "garden", "kitchen", "room", "stump", "treehill"]
iterations = 7000 #@param {type:"slider", min:1000, max:30000, step:1000}

if use_sample:
    source_path = f"/content/data/samples/{sample_scene}"
else:
    source_path = "/content/my_scene"

output_path = f"/content/output/{sample_scene if use_sample else 'my_scene'}"

print(f"\n🏋️ 3DGS学習開始")
print(f"   ソース: {source_path}")
print(f"   出力: {output_path}")
print(f"   イテレーション: {iterations}")
print("\n⏳ A100で約10-30分...\n")

!python train.py \
    -s {source_path} \
    -m {output_path} \
    --iterations {iterations}

print("\n✅ 学習完了！")

In [ ]:
#@title 🎥 **Step 6: レンダリング動画生成**
#@markdown 学習したモデルから360度回転動画を生成

sample_scene = "counter" #@param ["bicycle", "bonsai", "counter", "flowers", "garden", "kitchen", "room", "stump", "treehill"]
model_path = f"/content/output/{sample_scene}"

print(f"\n🎥 レンダリング中...")

!python render.py \
    -m {model_path}

print("\n✅ レンダリング完了！")
print(f"\n📁 出力: {model_path}/train/ours_7000/renders/")

In [ ]:
#@title 🎬 **Step 7: 動画に変換**
#@markdown レンダリングした画像を動画に

import glob
import imageio
from PIL import Image
import numpy as np

sample_scene = "counter" #@param {type:"string"}
model_path = f"/content/output/{sample_scene}"
render_path = f"{model_path}/train/ours_7000/renders/"
output_video = f"/content/{sample_scene}_3dgs.mp4"

# 画像を読み込んで動画に
images = sorted(glob.glob(f"{render_path}/*.png"))
print(f"\n📸 {len(images)}枚の画像を発見")

if len(images) > 0:
    frames = [np.array(Image.open(img)) for img in images]
    imageio.mimsave(output_video, frames, fps=30)
    print(f"\n✅ 動画保存: {output_video}")
    
    # 表示
    from IPython.display import HTML
    from base64 import b64encode
    mp4 = open(output_video, 'rb').read()
    data_url = f"data:video/mp4;base64,{b64encode(mp4).decode()}"
    display(HTML(f'<video width=800 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))
else:
    print("⚠️ レンダリング画像が見つかりません")

In [ ]:
#@title 💾 **Step 8: Google Driveに保存**
#@markdown 3DGSモデルをDriveにバックアップ

from google.colab import drive
drive.mount('/content/drive')

import shutil
from datetime import datetime

sample_scene = "counter" #@param {type:"string"}
model_path = f"/content/output/{sample_scene}"
backup_dir = f"/content/drive/MyDrive/IWASAKI_3DGS/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{sample_scene}"

# コピー
shutil.copytree(model_path, backup_dir)
print(f"\n✅ 保存完了: {backup_dir}")

# 動画もコピー
video_path = f"/content/{sample_scene}_3dgs.mp4"
if os.path.exists(video_path):
    shutil.copy(video_path, f"{backup_dir}/{sample_scene}_3dgs.mp4")
    print(f"   動画も保存")

---

## 📋 使い方まとめ

### サンプルで試す場合
1. Step 0-1: 環境構築
2. Step 2: サンプルダウンロード
3. Step 5: 学習（use_sample=True）
4. Step 6-7: レンダリング＆動画化

### 自分の写真を使う場合
1. Step 0-1: 環境構築
2. Step 3: 写真アップロード
3. Step 4: COLMAP（use_sample=False）
4. Step 5: 学習（use_sample=False）
5. Step 6-7: レンダリング＆動画化

---

## 🏠 内装業での活用

1. **施工現場を3D化** — お客様が「歩ける」完成予想
2. **Before/After比較** — 同じ視点から見比べ可能
3. **VR内覧会** — Quest等で没入体験
4. **ポートフォリオ** — 他社と差別化

---

*IWASAKI 3DGS STUDIO v1.0*